In [13]:
import re
import json
import ast
from typing import Union

from src.make_paths_relative_to_root import *

In [14]:
single_or_double_bracket_regex = re.compile(r"^(\[\[[^\[\]]+\]\])|^(\[[^\[\]]+\])")

In [15]:
DEBUG = True

In [16]:
def parse_model_output(output) -> Union[list, None]:
    triplet_string = getattr(
        re.match(single_or_double_bracket_regex, output), "string", None
    )

    if triplet_string is None:
        return None

    try:
        data = ast.literal_eval(triplet_string)
    except:
        return None

    if not isinstance(data, list):
        return None

    if isinstance(data[0], list):
        data = data[0]

    if len(data) != 3:
        return None

    return list(map(str, data))

In [17]:
with open("data/edc_baseline/extracted_relations.json") as f:
    relations = json.load(f)

In [18]:
with open("data/edc_baseline/reference.txt") as f:
    reference = list(map(ast.literal_eval, f.read().splitlines()))
    if DEBUG:
        reference = reference[:10]

In [19]:
print(len(relations["responses"]), len(reference))
assert len(relations["responses"]) == len(reference)

10 10


In [20]:
pairs = []
for resp, ref in zip(relations["responses"], reference):
    output = parse_model_output(resp)
    pairs.append((output, ref[0]))

In [21]:
DEBUG = True

if DEBUG:
    pairs = pairs[:50]

In [22]:
from xml.etree import ElementTree as ET


def stringify_triple(triple_list):
    return " | ".join(triple_list)


def create_xmls(pairs):
    cands_root = ET.Element("benchmark")
    cands_entries = ET.SubElement(cands_root, "entries")

    refs_root = ET.Element("benchmark")
    refs_entries = ET.SubElement(refs_root, "entries")

    for i, (output, ref) in enumerate(pairs):
        cand_entry = ET.SubElement(cands_entries, "entry")
        cand_entry.set("category", "Unknown")
        cand_entry.set("eid", f"Id{i}")

        gen_tripleset = ET.SubElement(cand_entry, "generatedtripleset")
        gtriple = ET.SubElement(gen_tripleset, "gtriple")

        if output is not None:
            try:
                gtriple.text = stringify_triple(output)
            except Exception as e:
                print(output)
                raise e

        ref_entry = ET.SubElement(refs_entries, "entry")
        ref_entry.set("category", "Unknown")
        ref_entry.set("eid", f"Id{i}")
        ref_entry.set("shape", "(X (X))")
        ref_entry.set("shape_type", "NA")
        ref_entry.set("size", "1")

        originaltripleset = ET.SubElement(ref_entry, "originaltripleset")
        otriple = ET.SubElement(originaltripleset, "otriple")
        otriple.text = stringify_triple(ref)

        modifiedtripleset = ET.SubElement(ref_entry, "modifiedtripleset")
        mtriple = ET.SubElement(modifiedtripleset, "mtriple")
        mtriple.text = stringify_triple(ref)

    return cands_root, refs_root


cands_root, refs_root = create_xmls(pairs)

In [23]:
cands_tree = ET.ElementTree(cands_root)
cands_tree.write("data/edc_baseline/cands.xml", encoding="utf-8", xml_declaration=True)

refs_tree = ET.ElementTree(refs_root)
refs_tree.write("data/edc_baseline/refs.xml", encoding="utf-8", xml_declaration=True)

In [24]:
!python src/evaluation/Evaluation_script_json.py data/edc_baseline/refs.xml data/edc_baseline/cands.xml data/edc_baseline/evaluation_results.json

src/evaluation/Evaluation_script_json.py:27: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  refssoup = BeautifulSoup(fp, 'lxml')
src/evaluation/Evaluation_script_json.py:60: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  candssoup = Beautiful